# Week 2 - Day 1: Descriptive Statistics

This notebook applies descriptive statistics to a vehicle sensor dataset and then uses the IQR method to review potential outliers.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

# Resolve the data file from several possible working directories.
possible_paths = []
for base in [Path.cwd(), *Path.cwd().parents[:3]]:
    possible_paths.extend([
        base / 'week1/day4/exp1_14drivers_14cars_dailyRoutes.csv',
        base / 'day4/exp1_14drivers_14cars_dailyRoutes.csv',
        base / 'week2/day1/../../week1/day4/exp1_14drivers_14cars_dailyRoutes.csv',
    ])

for candidate in possible_paths:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError('Could not find the vehicle dataset.')

data = pd.read_csv(data_path, low_memory=False)

selected_columns = ['MARK', 'MODEL', 'CAR_YEAR', 'SPEED', 'ENGINE_COOLANT_TEMP', 'ENGINE_RPM', 'AIR_INTAKE_TEMP']
cars = data[selected_columns].copy()

print(cars.head())
cars.info()


        MARK  MODEL  CAR_YEAR  SPEED  ENGINE_COOLANT_TEMP  ENGINE_RPM  \
0  chevrolet  agile    2011.0    0.0                 80.0      1009.0   
1  chevrolet  agile    2011.0    0.0                 80.0      1003.0   
2  chevrolet  agile    2011.0    0.0                 80.0       995.0   
3  chevrolet  agile    2011.0    0.0                 80.0      1004.0   
4  chevrolet  agile    2011.0    0.0                 80.0      1005.0   

   AIR_INTAKE_TEMP  
0             59.0  
1             59.0  
2             59.0  
3             60.0  
4             60.0  
<class 'pandas.DataFrame'>
RangeIndex: 47514 entries, 0 to 47513
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   MARK                 47459 non-null  str    
 1   MODEL                47459 non-null  str    
 2   CAR_YEAR             47459 non-null  float64
 3   SPEED                46529 non-null  float64
 4   ENGINE_COOLANT_TEMP  33964 non-nu

In [2]:
cars_with_speed = cars.dropna(subset=['SPEED']).copy()
speed = cars_with_speed['SPEED']

print('Rows with available speed values:', len(cars_with_speed))
print('Mean speed:', speed.mean())
print('Median speed:', speed.median())
print('Mode speed:', speed.mode().iloc[0])
print('Range speed:', speed.max() - speed.min())
print('Variance speed:', speed.var())
print('Standard deviation speed:', speed.std())

Rows with available speed values: 46529
Mean speed: 24.743751208923467
Median speed: 14.0
Mode speed: 0.0
Range speed: 143.0
Variance speed: 858.1394374497937
Standard deviation speed: 29.29401709308223


The speed values were reviewed after removing rows with missing speed measurements. The next step compares the spread of the data before and after filtering out potential outliers.

In [3]:
q1 = speed.quantile(0.25)
q3 = speed.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outlier_mask = ~speed.between(lower_bound, upper_bound)
outlier_count = outlier_mask.sum()
outlier_percentage = outlier_count / len(cars_with_speed) * 100

filtered_cars = cars_with_speed[speed.between(lower_bound, upper_bound)].copy()

print('Q1:', q1)
print('Q3:', q3)
print('IQR:', iqr)
print('Lower bound:', lower_bound)
print('Upper bound:', upper_bound)
print('Number of potential outliers:', outlier_count)
print('Outlier percentage:', f'{outlier_percentage:.2f}%')
print('Filtered shape:', filtered_cars.shape)

Q1: 0.0
Q3: 42.0
IQR: 42.0
Lower bound: -63.0
Upper bound: 105.0
Number of potential outliers: 744
Outlier percentage: 1.60%
Filtered shape: (45785, 7)


The IQR method flagged observations outside the calculated bounds as potential outliers. The filtered dataset is used to compare the effect of these observations on the statistical measures. Values outside the bounds are not automatically considered invalid because they may represent genuine high-speed observations.

In [4]:
print('Mean after filtering:', filtered_cars['SPEED'].mean())
print('Median after filtering:', filtered_cars['SPEED'].median())
print('Variance after filtering:', filtered_cars['SPEED'].var())
print('Standard deviation after filtering:', filtered_cars['SPEED'].std())
print('Range after filtering:', filtered_cars['SPEED'].max() - filtered_cars['SPEED'].min())

Mean after filtering: 23.265763896472645
Median after filtering: 13.0
Variance after filtering: 734.3149612289822
Standard deviation after filtering: 27.0982464604074
Range after filtering: 105.0


In [5]:
plt.figure(figsize=(8, 5))
sns = plt
plt.boxplot(speed, vert=False)
plt.title('Vehicle Speed Distribution')
plt.xlabel('Speed')
plt.tight_layout()
plt.show()

/tmp/ipykernel_598954/3648952877.py:3: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  plt.boxplot(speed, vert=False)
/tmp/ipykernel_598954/3648952877.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The box plot shows the overall spread of the speed values and highlights the observations that fall outside the IQR-based bounds. The comparison before and after filtering shows that the outliers influenced the range and spread more than the central tendency measures.